In [ ]:
mport torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd


class ProjectionHead(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=512, output_dim=256):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        x = self.net(x)

        # Normalização L2
        x = F.normalize(x, p=2, dim=1)

        return x

In [ ]:
class MultimodalModel(nn.Module):
    def __init__(
        self,
        image_dim=1024,
        text_dim=1024,
        hidden_dim=512,
        embedding_dim=256
    ):
        super().__init__()

        self.image_projection = ProjectionHead(
            image_dim,
            hidden_dim,
            embedding_dim
        )

        self.text_projection = ProjectionHead(
            text_dim,
            hidden_dim,
            embedding_dim
        )

    def forward(self, image, text):

        image_embedding = self.image_projection(image)
        text_embedding = self.text_projection(text)

        return image_embedding, text_embedding

In [ ]:
def contrastive_loss(image_z, text_z, temperature=0.07):

    similarity = image_z @ text_z.T

    similarity = similarity / temperature

    labels = torch.arange(
        image_z.size(0),
        device=image_z.device
    )

    loss_image_to_text = F.cross_entropy(
        similarity,
        labels
    )

    loss_text_to_image = F.cross_entropy(
        similarity.T,
        labels
    )

    loss = (
        loss_image_to_text +
        loss_text_to_image
    ) / 2

    return loss

In [ ]:
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ==================================================
# DATASET
# ==================================================

class EmbeddingDataset(Dataset):
    """Pares (embedding de imagem, embedding de texto) já pré-computados."""

    def __init__(self, image_embeddings, text_embeddings):
        assert len(image_embeddings) == len(text_embeddings), \
            "Imagens e textos precisam ter o mesmo número de amostras."

        # Converte uma única vez (e não a cada __getitem__)
        self.images = torch.as_tensor(image_embeddings, dtype=torch.float32)
        self.texts = torch.as_tensor(text_embeddings, dtype=torch.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.texts[idx]

train_image_embeddings = 
train_text_embeddings
val_image_embeddings 
val_text_embeddings
train_dataset = EmbeddingDataset(train_image_embeddings, train_text_embeddings)
val_dataset = EmbeddingDataset(val_image_embeddings, val_text_embeddings)

# Os dados já estão em memória: workers só adicionam overhead.
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    drop_last=True,      # evita último batch minúsculo (ruim para loss contrastiva)
    pin_memory=True,
)

# shuffle=False é obrigatório: o par correto de i é o índice i
val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    pin_memory=True,
)


# ==================================================
# MÉTRICAS DE RETRIEVAL
# ==================================================

@torch.no_grad()
def positive_ranks(queries, gallery, chunk_size=2048):
    """
    Posição (1 = melhor) do par correto de cada query dentro da galeria.
    Processa em blocos para não montar a matriz N x N inteira na memória.
    Empates contam contra o modelo (ranking pessimista).
    """
    ranks = []

    for start in range(0, queries.size(0), chunk_size):
        sim = queries[start:start + chunk_size] @ gallery.T
        rows = torch.arange(sim.size(0), device=sim.device)
        positives = sim[rows, rows + start].unsqueeze(1)
        ranks.append((sim >= positives).sum(dim=1))

    return torch.cat(ranks)


@torch.no_grad()
def retrieval_metrics(image_embeddings, text_embeddings, ks=(1, 5, 10)):
    """Recall@K e MRR para Image->Text e Text->Image, a partir de um único ranking."""

    # Garante similaridade de cosseno mesmo que o modelo não normalize
    image_embeddings = F.normalize(image_embeddings, dim=-1)
    text_embeddings = F.normalize(text_embeddings, dim=-1)

    ranks = {
        "i2t": positive_ranks(image_embeddings, text_embeddings),
        "t2i": positive_ranks(text_embeddings, image_embeddings),
    }

    metrics = {}

    for direction, r in ranks.items():
        for k in ks:
            metrics[f"r{k}_{direction}"] = (r <= k).float().mean().item()
        metrics[f"mrr_{direction}"] = (1.0 / r.float()).mean().item()

    for k in ks:
        metrics[f"r{k}_mean"] = (metrics[f"r{k}_i2t"] + metrics[f"r{k}_t2i"]) / 2
    metrics["mrr_mean"] = (metrics["mrr_i2t"] + metrics["mrr_t2i"]) / 2

    return metrics


@torch.no_grad()
def validate(model, val_loader, device):
    model.eval()

    total_loss = torch.zeros((), device=device)
    all_image, all_text = [], []

    for image, text in val_loader:
        image = image.to(device, non_blocking=True)
        text = text.to(device, non_blocking=True)

        image_z, text_z = model(image, text)

        total_loss += contrastive_loss(image_z, text_z)
        all_image.append(image_z)
        all_text.append(text_z)

    metrics = retrieval_metrics(torch.cat(all_image), torch.cat(all_text))
    metrics["loss"] = (total_loss / len(val_loader)).item()

    return metrics


# ==================================================
# CONFIGURAÇÃO DO TREINAMENTO
# ==================================================

model = MultimodalModel().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_epochs = 20
patience = 5
min_delta = 1e-4

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

best_recall = -float("inf")
best_epoch = 0
epochs_without_improvement = 0
history = []


# ==================================================
# TREINAMENTO
# ==================================================

for epoch in range(1, num_epochs + 1):

    model.train()
    total_train_loss = torch.zeros((), device=device)

    for image, text in train_loader:
        image = image.to(device, non_blocking=True)
        text = text.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        image_z, text_z = model(image, text)
        loss = contrastive_loss(image_z, text_z)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.detach()   # evita sincronizar GPU->CPU a cada passo

    scheduler.step()

    train_loss = (total_train_loss / len(train_loader)).item()
    val = validate(model, val_loader, device)
    history.append({"epoch": epoch, "train_loss": train_loss, **val})

    print(f"\nEpoch {epoch}/{num_epochs} | LR: {scheduler.get_last_lr()[0]:.2e}")
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val['loss']:.4f}")

    for label, key in (("Image → Text", "i2t"), ("Text → Image", "t2i"), ("Retrieval médio", "mean")):
        print(
            f"{label:<16}| "
            f"R@1: {val[f'r1_{key}']:.4f} | "
            f"R@5: {val[f'r5_{key}']:.4f} | "
            f"R@10: {val[f'r10_{key}']:.4f} | "
            f"MRR: {val[f'mrr_{key}']:.4f}"
        )

    # ----------------------------------------------
    # Seleção do melhor modelo + early stopping
    # ----------------------------------------------

    if val["r1_mean"] > best_recall + min_delta:
        best_recall = val["r1_mean"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                **{k: val[k] for k in ("loss",
                                        "r1_i2t",
                                        "r5_i2t",
                                        "r10_i2t",
                                        "mrr_i2t",
                                        "r1_t2i",
                                        "r5_t2i",
                                        "r10_t2i",
                                        "mrr_t2i",
                                        "r1_mean",
                                        "r5_mean",
                                        "r10_mean",
                                        "mrr_mean")
                },
            },
            "best_model.pt",
        )
        print(f"✓ Novo melhor modelo! Val R@1 médio = {best_recall:.4f}")

    else:
        epochs_without_improvement += 1
        print(f"Sem melhoria: {epochs_without_improvement}/{patience}")

        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping. Melhor época: {best_epoch} | Val R@1 médio: {best_recall:.4f}")
            break


# Recarrega o melhor checkpoint ao final (o modelo em memória é o da última época)
checkpoint = torch.load("best_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])